<!--
SPDX-FileCopyrightText: Copyright (c) 2025-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0
-->
# 🕵️ Choosing a Replacement Strategy

Four [replace mode](../../concepts/replace/) strategies compared side-by-side on the same data.

| Strategy | What it does |
|----------|-------------|
| **Substitute** | LLM-generated contextual replacements |
| **Redact** | Label-based markers (`[REDACTED_FIRST_NAME]`) |
| **Annotate** | Tags entities but keeps original text |
| **Hash** | Deterministic hash digest |

#### 📚 What you'll learn

- Compare **Redact**, **Annotate**, **Hash**, and **Substitute** on the same input
- Customize output formats with `format_template`
- Understand which strategy fits your use case (readability, determinism, privacy)

> **Tip:** First time running notebooks? Start with
> [setup instructions](https://nvidia-nemo.github.io/Anonymizer/latest/tutorials/).

## ⚙️ Setup

- Install the notebook extra, then provide credentials for the configured external LLM providers.
- `create_anonymizer()` starts pinned GLiNER2 locally and selects CUDA, MPS, or CPU automatically.
- The default external LLM models currently use [OpenRouter](https://openrouter.ai); its terms and privacy practices apply.

> **Data boundary:** GLiNER2 detection runs locally in this notebook environment. LLM-assisted validation,
> augmentation, replacement, rewriting, repair, and evaluation use configured external hosts and may send
> them original or tagged input text. Do not treat this configuration as an all-local privacy boundary.
- `configure_logging(LoggingConfig.default())` keeps logs at INFO. Switch to `LoggingConfig.debug()` when troubleshooting.

In [1]:
import getpass
import os
import subprocess
import sys

package_spec = os.getenv("ANONYMIZER_NOTEBOOK_PACKAGE", "nemo-anonymizer[notebooks]")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package_spec])

0

In [2]:
from anonymizer.notebooks import required_api_key_environment_variables

for variable in required_api_key_environment_variables():
    key = getpass.getpass(f"Enter {variable}: ").strip()
    if not key:
        raise RuntimeError(f"{variable} is required by the configured external model providers.")
    os.environ[variable] = key

In [3]:
from anonymizer import (
    Annotate,
    AnonymizerConfig,
    AnonymizerInput,
    Hash,
    LoggingConfig,
    Redact,
    Substitute,
    configure_logging,
)
from anonymizer.notebooks import create_anonymizer, stop_local_runtime

configure_logging(LoggingConfig.default())

In [4]:
anonymizer = create_anonymizer()

[00:19:51] [INFO] 🔧 Anonymizer initialized with 5 model configs


[00:19:51] [INFO]   |-- 🔎 detector:  local-gliner2-pii


[00:19:51] [INFO]   |-- ✅ validator: gpt-oss-120b


[00:19:51] [INFO]   |-- 🧩 augmenter: gpt-oss-120b


GLiNER2 ready: model=fastino/gliner2-privacy-filter-PII-multi revision=59894c087cb2923b01f337d4ee72f6ff84d5bdd6 device=mps endpoint=http://127.0.0.1:54611/v1


## 📦 Input data

- We use the same biographies dataset throughout so each strategy is compared
  on identical input.

In [5]:
input_data = AnonymizerInput(
    source="https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv",
    text_column="biography",
    data_summary="Biographical profiles",
)

## 🔄 Substitute

- Uses an LLM to generate contextually appropriate synthetic replacements.
  - The LLM considers the full document context matching names with emails, cities to states, etc.
- Customize with `instructions` to steer the LLM's replacement choices.

In [6]:
substitute_config = AnonymizerConfig(replace=Substitute())

substitute_preview = anonymizer.preview(
    config=substitute_config,
    data=input_data,
    num_records=3,
)

[00:19:51] [INFO] 👀 Preview mode: 📂 Loaded 3 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[00:19:51] [INFO] 🔍 Running entity detection on 3 records


[00:19:51] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[00:21:28] [INFO]   |-- 📋 Detection complete — 84 entities found across 3 records (0 failed) [97.0s]


[00:21:28] [INFO]   |-- labels: first_name=23, occupation=6, age=5, field_of_study=5, company_name=5, city=4, organization_name=4, degree=4, university=4, last_name=3, state=3, language=3, political_view=3, religious_belief=3, place_name=2, race_ethnicity=2, street_address=2, nationality=1, date_of_birth=1, landmark=1


[00:21:28] [INFO] 🔄 Running Substitute replacement


[00:22:12] [INFO]   |-- 📋 Replacement complete (0 failed) [44.0s]


[00:22:12] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


In [7]:
substitute_preview.display_record(0)

Original,Label,Replacement
40‑year‑old,age,52-year-old
Aria,first_name,Jenna
Bobby,first_name,Ethan
Christian Democrat,political_view,Libertarian
Colorado,state,Oregon
Colorado Veterinary Clinic,organization_name,Willamette Animal Clinic
DVM,degree,Doctor of Medicine (MD)
Denver,city,Portland
English,language,Spanish
Jefferson High,organization_name,Lincoln High School


### Custom instructions

- Pass `instructions` to guide the LLM -- e.g. keep replacements within
  a specific region, culture, or naming convention.

In [8]:
substitute_custom_config = AnonymizerConfig(
    replace=Substitute(instructions="Use only Japanese names and locations for all replacements.")
)
substitute_custom_preview = anonymizer.preview(
    config=substitute_custom_config,
    data=input_data,
    num_records=3,
)
substitute_custom_preview.display_record(0)

[00:22:14] [INFO] 👀 Preview mode: 📂 Loaded 3 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[00:22:14] [INFO] 🔍 Running entity detection on 3 records


[00:22:14] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[00:23:33] [INFO]   |-- 📋 Detection complete — 84 entities found across 3 records (0 failed) [78.6s]


[00:23:33] [INFO]   |-- labels: first_name=23, occupation=6, company_name=6, age=5, field_of_study=5, city=4, organization_name=4, degree=4, university=4, last_name=3, state=3, language=3, political_view=3, place_name=2, race_ethnicity=2, street_address=2, religious_belief=2, nationality=1, date_of_birth=1, landmark=1


[00:23:33] [INFO] 🔄 Running Substitute replacement


[00:24:01] [INFO]   |-- 📋 Replacement complete (0 failed) [28.5s]


[00:24:01] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
40‑year‑old,age,45‑year‑old
Aria,first_name,Yui
Bobby,first_name,Haruto
Christian Democrat,political_view,Liberal Democratic Party member
Colorado,state,Hokkaido
Colorado Veterinary Clinic,organization_name,Hokkaido Veterinary Center
DVM,degree,Bachelor of Science
Denver,city,Sapporo
English,language,Japanese
Jefferson High,organization_name,Kobe High School


## 🚫 Redact

- Replaces each entity with a label-based marker. Default: `[REDACTED_FIRST_NAME]`.
- Customize with `Redact(format_template=...)`.

In [9]:
redact_config = AnonymizerConfig(replace=Redact())

redact_preview = anonymizer.preview(
    config=redact_config,
    data=input_data,
    num_records=3,
)

redact_preview.display_record(0)

[00:24:02] [INFO] 👀 Preview mode: 📂 Loaded 3 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[00:24:02] [INFO] 🔍 Running entity detection on 3 records


[00:24:02] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[00:25:20] [INFO]   |-- 📋 Detection complete — 85 entities found across 3 records (0 failed) [77.7s]


[00:25:20] [INFO]   |-- labels: first_name=23, occupation=6, age=5, university=5, field_of_study=5, organization_name=5, company_name=5, city=4, last_name=3, state=3, language=3, political_view=3, degree=3, place_name=2, race_ethnicity=2, street_address=2, religious_belief=2, nationality=1, education_level=1, date_of_birth=1, landmark=1


[00:25:20] [INFO] 🔄 Running Redact replacement


[00:25:20] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[00:25:20] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
Bobby,first_name,[REDACTED_FIRST_NAME]
Watford,last_name,[REDACTED_LAST_NAME]
40‑year‑old,age,[REDACTED_AGE]
Mexican,nationality,[REDACTED_NATIONALITY]
veterinarian,occupation,[REDACTED_OCCUPATION]
Denver,city,[REDACTED_CITY]
Colorado,state,[REDACTED_STATE]
Jefferson High,university,[REDACTED_UNIVERSITY]
DVM,education_level,[REDACTED_EDUCATION_LEVEL]
University of Colorado Boulder,university,[REDACTED_UNIVERSITY]


### Custom template

- `format_template="***"` replaces every entity with the same constant.

In [10]:
custom_config = AnonymizerConfig(replace=Redact(format_template="***"))

custom_preview = anonymizer.preview(
    config=custom_config,
    data=input_data,
    num_records=3,
)

custom_preview.display_record(0)

[00:25:21] [INFO] 👀 Preview mode: 📂 Loaded 3 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[00:25:21] [INFO] 🔍 Running entity detection on 3 records


[00:25:21] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[00:26:20] [INFO]   |-- 📋 Detection complete — 85 entities found across 3 records (0 failed) [58.9s]


[00:26:20] [INFO]   |-- labels: first_name=23, occupation=6, age=5, organization_name=5, field_of_study=5, company_name=5, city=4, university=4, last_name=3, state=3, language=3, political_view=3, degree=3, religious_belief=3, place_name=2, race_ethnicity=2, street_address=2, nationality=1, education_level=1, date_of_birth=1, landmark=1


[00:26:20] [INFO] 🔄 Running Redact replacement


[00:26:20] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[00:26:20] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
Bobby,first_name,***
Watford,last_name,***
40‑year‑old,age,***
Mexican,nationality,***
veterinarian,occupation,***
Denver,city,***
Colorado,state,***
Jefferson High,organization_name,***
DVM,education_level,***
University of Colorado Boulder,university,***


## 🏷️ Annotate

- Tags each entity with its label but keeps the original text visible.
  Default: `<Alice, first_name>`.
- Customize with `format_template` -- must include `{text}` and `{label}`,
  e.g. `Annotate(format_template="<{text}-|-{label}>")`.

In [11]:
annotate_config = AnonymizerConfig(replace=Annotate())

annotate_preview = anonymizer.preview(
    config=annotate_config,
    data=input_data,
    num_records=3,
)

annotate_preview.display_record(0)

[00:26:21] [INFO] 👀 Preview mode: 📂 Loaded 3 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[00:26:21] [INFO] 🔍 Running entity detection on 3 records


[00:26:21] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[00:26:58] [INFO]   |-- 📋 Detection complete — 85 entities found across 3 records (0 failed) [36.7s]


[00:26:58] [INFO]   |-- labels: first_name=23, occupation=6, age=5, organization_name=5, field_of_study=5, company_name=5, city=4, degree=4, university=4, last_name=3, state=3, language=3, political_view=3, religious_belief=3, place_name=2, race_ethnicity=2, street_address=2, nationality=1, date_of_birth=1, landmark=1


[00:26:58] [INFO] 🔄 Running Annotate replacement


[00:26:58] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[00:26:58] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
Bobby,first_name,"<Bobby, first_name>"
Watford,last_name,"<Watford, last_name>"
40‑year‑old,age,"<40‑year‑old, age>"
Mexican,nationality,"<Mexican, nationality>"
veterinarian,occupation,"<veterinarian, occupation>"
Denver,city,"<Denver, city>"
Colorado,state,"<Colorado, state>"
Jefferson High,organization_name,"<Jefferson High, organization_name>"
DVM,degree,"<DVM, degree>"
University of Colorado Boulder,university,"<University of Colorado Boulder, university>"


### Custom template

- Override the default format with any string containing `{text}` and `{label}`.

In [12]:
annotate_custom_config = AnonymizerConfig(replace=Annotate(format_template="<{text}-|-{label}>"))
annotate_custom_preview = anonymizer.preview(
    config=annotate_custom_config,
    data=input_data,
    num_records=3,
)
annotate_custom_preview.display_record(0)

[00:26:59] [INFO] 👀 Preview mode: 📂 Loaded 3 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[00:26:59] [INFO] 🔍 Running entity detection on 3 records


[00:26:59] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[00:27:58] [INFO]   |-- 📋 Detection complete — 83 entities found across 3 records (0 failed) [59.4s]


[00:27:58] [INFO]   |-- labels: first_name=23, occupation=6, age=5, field_of_study=5, company_name=5, city=4, organization_name=4, degree=4, university=4, last_name=3, state=3, language=3, political_view=3, place_name=2, race_ethnicity=2, street_address=2, religious_belief=2, nationality=1, date_of_birth=1, landmark=1


[00:27:58] [INFO] 🔄 Running Annotate replacement


[00:27:58] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[00:27:58] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
Bobby,first_name,<Bobby-|-first_name>
Watford,last_name,<Watford-|-last_name>
40‑year‑old,age,<40‑year‑old-|-age>
Mexican,nationality,<Mexican-|-nationality>
veterinarian,occupation,<veterinarian-|-occupation>
Denver,city,<Denver-|-city>
Colorado,state,<Colorado-|-state>
Jefferson High,organization_name,<Jefferson High-|-organization_name>
DVM,degree,<DVM-|-degree>
University of Colorado Boulder,university,<University of Colorado Boulder-|-university>


## #️⃣ Hash

- Deterministic -- same input always produces the same hash.
- Customize with `format_template` (must include `{digest}`),
  `algorithm` (`sha256`/`sha1`/`md5`), and `digest_length` (6-64 characters).

In [13]:
hash_config = AnonymizerConfig(replace=Hash())

hash_preview = anonymizer.preview(
    config=hash_config,
    data=input_data,
    num_records=3,
)

hash_preview.display_record(0)

[00:27:59] [INFO] 👀 Preview mode: 📂 Loaded 3 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[00:27:59] [INFO] 🔍 Running entity detection on 3 records


[00:27:59] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[00:28:36] [INFO]   |-- 📋 Detection complete — 83 entities found across 3 records (0 failed) [37.1s]


[00:28:36] [INFO]   |-- labels: first_name=23, occupation=6, age=5, university=5, field_of_study=5, company_name=5, city=4, last_name=3, state=3, language=3, organization_name=3, political_view=3, degree=3, place_name=2, race_ethnicity=2, street_address=2, religious_belief=2, nationality=1, education_level=1, date_of_birth=1, landmark=1


[00:28:36] [INFO] 🔄 Running Hash replacement


[00:28:36] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[00:28:36] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
Bobby,first_name,<HASH_FIRST_NAME_4a70dab2cb4d>
Watford,last_name,<HASH_LAST_NAME_e2efa8a62600>
40‑year‑old,age,<HASH_AGE_c19577d3d98d>
Mexican,nationality,<HASH_NATIONALITY_d108dfd1df5c>
veterinarian,occupation,<HASH_OCCUPATION_52a469e4d8e9>
Denver,city,<HASH_CITY_fcdeb8c07d4a>
Colorado,state,<HASH_STATE_4ae62bf4e804>
Jefferson High,university,<HASH_UNIVERSITY_39dde416149c>
DVM,education_level,<HASH_EDUCATION_LEVEL_d44ae5e206d1>
University of Colorado Boulder,university,<HASH_UNIVERSITY_bca201129c41>


### Custom template

- Override the algorithm, digest length, and output format.

In [14]:
hash_custom_config = AnonymizerConfig(replace=Hash(algorithm="md5", digest_length=8, format_template="[{digest}]"))
hash_custom_preview = anonymizer.preview(
    config=hash_custom_config,
    data=input_data,
    num_records=3,
)
hash_custom_preview.display_record(0)

[00:28:37] [INFO] 👀 Preview mode: 📂 Loaded 3 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')


[00:28:37] [INFO] 🔍 Running entity detection on 3 records


[00:28:37] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)


[00:29:55] [INFO]   |-- 📋 Detection complete — 84 entities found across 3 records (0 failed) [78.0s]


[00:29:55] [INFO]   |-- labels: first_name=22, occupation=6, age=5, organization_name=5, field_of_study=5, company_name=5, city=4, degree=4, university=4, last_name=3, state=3, language=3, political_view=3, religious_belief=3, place_name=2, race_ethnicity=2, street_address=2, nationality=1, date_of_birth=1, landmark=1


[00:29:55] [INFO] 🔄 Running Hash replacement


[00:29:55] [INFO]   |-- 📋 Replacement complete (0 failed) [0.0s]


[00:29:55] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Original,Label,Replacement
Bobby,first_name,[657b3da9]
Watford,last_name,[6e424e2c]
40‑year‑old,age,[7d5ce251]
Mexican,nationality,[a0e769d8]
veterinarian,occupation,[84c99b4a]
Denver,city,[67100af8]
Colorado,state,[15e49475]
Jefferson High,organization_name,[27c56955]
DVM,degree,[47211f54]
University of Colorado Boulder,university,[e2b97348]


## 📊 (Optional) Evaluate each strategy

- `evaluate()` is a separate, opt-in step that scores the output with LLM-as-judge metrics. Which metrics fire depends on the strategy:
  - **Substitute** → 4 metrics (Detection Validity + Type Fidelity + Relational Consistency + Attribute Fidelity).
  - **Redact / Annotate / Hash** → Detection Validity only (no replacement map to score type/relational/attribute against).
- Below shows it on the Substitute preview to surface all four; the same call works on `redact_preview`, `annotate_preview`, or `hash_preview`.

In [15]:
substitute_evaluated = anonymizer.evaluate(substitute_preview)
substitute_evaluated.display_record(0)

[00:29:55] [INFO] 🧪 Running Substitute evaluation on 3 records


[00:29:55] [INFO]   |-- ⚖️ Running replace judges


[00:32:47] [INFO]   |-- 📋 Replace judges complete [171.3s]


[00:32:47] [INFO] 🎉 Evaluation complete — 3 records processed [171.3s]


## ⏭️ Next steps

- **[🕵️ Inspecting Detected Entities](../02_inspecting_detected_entities/)** --
  dig into what the detection pipeline found and debug quality.
- **[✏️ Rewriting Biographies](../04_rewriting_biographies/)** --
  generate privacy-safe paraphrases instead of token-level replacements.
- **[⚖️ Rewriting Legal Documents](../05_rewriting_legal_documents/)** --
  rewrite legal text with domain-specific privacy goals.

In [16]:
stop_local_runtime()